In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

In [2]:
transactions = pd.read_csv(
    '../data/bronze/transactions_history_final.csv',
    low_memory=False
)

outlet_master = pd.read_csv('../data/bronze/outlet_master.csv')

outlet_coordinates = pd.read_csv('../data/bronze/outlet_coordinates.csv')

seasonality = pd.read_csv(
    '../data/bronze/distributor_seasonality_details.csv'
)

holidays = pd.read_csv('../data/bronze/holiday_list.csv')

REJECT LOGGER FUNCTION

In [3]:
def create_rejected_store(df, reason):

    rejected = df.copy()

    rejected['rejection_reason'] = reason

    rejected['rejected_timestamp'] = datetime.now()

    return rejected

REUSABLE DATA QUALITY CHECKS

CHECK 1 — NULL CHECK

In [4]:
def null_check(df, required_columns):

    mask = df[required_columns].isnull().any(axis=1)

    failed = df[mask]

    passed = df[~mask]

    return passed, failed

RUN NULL CHECK

In [6]:
required_columns = [
    'Outlet_ID',
    'Distributor_ID',
    'SKU_ID',
    'Volume_Liters',
    'Total_Bill_Value'
]

transactions_clean, transactions_nulls = null_check(
    transactions,
    required_columns
)

transactions_nulls = create_rejected_store(
    transactions_nulls,
    'NULL_CHECK_FAILED'
)

print(transactions_nulls.shape)

(0, 9)


DUPLICATE CHECK

In [7]:
def duplicate_check(df, subset_cols):

    mask = df.duplicated(
        subset=subset_cols,
        keep='first'
    )

    failed = df[mask]

    passed = df[~mask]

    return passed, failed

In [8]:
dup_cols = [
    'Outlet_ID',
    'Year',
    'Month',
    'Distributor_ID',
    'SKU_ID',
    'Volume_Liters',
    'Total_Bill_Value'
]

transactions_clean, duplicate_rows = duplicate_check(
    transactions_clean,
    dup_cols
)

duplicate_rows = create_rejected_store(
    duplicate_rows,
    'DUPLICATE_RECORD'
)

print(duplicate_rows.shape)

(0, 9)


VALUE RANGE CHECKS

In [9]:
def range_check(df, column, min_val=None, max_val=None):

    mask = pd.Series(False, index=df.index)

    if min_val is not None:
        mask |= df[column] < min_val

    if max_val is not None:
        mask |= df[column] > max_val

    failed = df[mask]

    passed = df[~mask]

    return passed, failed

CHECK NEGATIVE VOLUMES

In [11]:
transactions_clean, negative_volume = range_check(
    transactions_clean,
    'Volume_Liters',
    min_val=0
)

negative_volume = create_rejected_store(
    negative_volume,
    'NEGATIVE_VOLUME'
)

print(negative_volume.shape)

(4753, 9)


CHECK NEGATIVE BILL VALUES

In [12]:
transactions_clean, negative_bill = range_check(
    transactions_clean,
    'Total_Bill_Value',
    min_val=0
)

negative_bill = create_rejected_store(
    negative_bill,
    'NEGATIVE_BILL_VALUE'
)

print(negative_bill.shape)

(0, 9)


ZERO VOLUME GHOSTS

In [13]:
zero_volume = transactions_clean[
    transactions_clean['Volume_Liters'] == 0
]

zero_volume = create_rejected_store(
    zero_volume,
    'ZERO_VOLUME_GHOST'
)

transactions_clean = transactions_clean[
    transactions_clean['Volume_Liters'] != 0
]

print(zero_volume.shape)

(100, 9)


REFERENTIAL INTEGRITY CHECKS